In [ ]:
import pandas as pd


# df = pd.read_csv('Data\gefs\klga.csv', nrows=50000).set_index(['init_time', 'valid_time'])
df = pd.read_parquet(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\klga.parquet').set_index(['init_time', 'valid_time'])

cols = df.isna().sum().sort_values(ascending=False)
drop_cols = cols[cols > len(df)*0.1].index
df.drop(drop_cols, axis=1, inplace=True)


df_clean = df.dropna()



In [ ]:
df_clean.to_csv(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\klga_clean.csv')


In [ ]:
from scipy.constants import convert_temperature
import pandas as pd

df = pd.read_parquet(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\klga_clean.parquet', engine='pyarrow').reset_index().set_index('valid_time')#.drop('index', axis=1)
y = pd.read_csv(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\nyc.csv', index_col=0).y
y.index = pd.to_datetime(y.index, utc=True)
y = y.tz_convert(None)


test = df.reset_index().set_index(['init_time', 'valid_time', 'lead_time'])[['temperature_2m','ensemble_member']].pivot(columns='ensemble_member')

test.columns = test.columns.get_level_values(1)
test = test.reset_index(['lead_time', 'init_time'])
test['y'] = y
test.iloc[:,2:-1] = test.iloc[:,2:-1].apply(convert_temperature, args=('celsius', 'fahrenheit'))
test.to_parquet(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\emos_ready.parquet', engine='pyarrow')


In [ ]:

def add_calendar_features(df):
    """
    Adds calendar and seasonal features to a DataFrame
    based on its DatetimeIndex.
    """

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame index must be a pandas DatetimeIndex.")

    # Season function
    def get_season(date):
        md = date.month * 100 + date.day
        if 321 <= md <= 620:
            return "spring"
        elif 621 <= md <= 922:
            return "summer"
        elif 923 <= md <= 1220:
            return "autumn"
        else:
            return "winter"

    idx = df.index

    df = df.copy()  # avoid modifying original

    df['year']         = idx.year
    df['month']        = idx.month
    df['month_name']   = idx.month_name()
    df['day']          = idx.day
    df['weekday']      = idx.weekday
    df['weekday_name'] = idx.day_name()
    df['hour']         = idx.hour
    df['dayofyear']    = idx.dayofyear
    df['week']         = idx.isocalendar().week
    df['season']       = idx.to_series().apply(get_season)

    return df

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\klga_clean.parquet', engine='pyarrow').reset_index().set_index('valid_time')#.drop('index', axis=1)
#df = pd.read_csv(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\klga_clean.csv', nrows=100000).reset_index().set_index('valid_time').drop('index', axis=1)
y = pd.read_csv(r'C:\Users\rasmu\Desktop\Candidate\Data\gefs\nyc.csv', index_col=0).y
y.index = pd.to_datetime(y.index, utc=True)
y = y.tz_convert(None)

df = df[df.lead_time == '0 days 03:00:00']

#df = df.join(y).sort_index()
#df = df.drop(df.select_dtypes('object').columns.tolist(), axis = 1)



In [ ]:
from sklearn.preprocessing import LabelEncoder
from scipy.constants import convert_temperature
from sklearn.model_selection import train_test_split

aa = df.reset_index()
drop_cols = """categorical_freezing_rain_surface
categorical_ice_pellets_surface
categorical_rain_surface
categorical_snow_surface
precipitation_surface
pressure_surface
wind_u_100m
wind_v_100m
latitude
longitude
spatial_ref
lead_time
expected_forecast_length""".splitlines()



bb = aa.drop(drop_cols, axis=1)
bb = bb.set_index(['init_time', 'valid_time']).pivot(columns='ensemble_member').reset_index().set_index('valid_time').drop('init_time', axis=1)
bb.columns = [(x,y)  if y != '' else x for x,y in zip(bb.columns.get_level_values(0), bb.columns.get_level_values(1))]

lead_time = aa.set_index('valid_time').lead_time

bb['y'] = bb.index.map(y)
bb = bb.join(lead_time)
bb.index = pd.to_datetime(bb.index)
bb.sort_index(inplace=True)

bb = add_calendar_features(bb)

cat_cols = bb.select_dtypes('object').columns
for col in cat_cols:
    le = LabelEncoder()
    bb[col] = le.fit_transform(bb[col].astype(str))


bb.dropna(inplace=True)
temps = [col for col in bb.columns if 'temperature_2m' == col[0]]

bb[temps] = bb[temps].apply(convert_temperature, args=('celsius', 'fahrenheit'))
# bb = bb.drop_duplicates()


X = bb.drop('y', axis=1)
y = bb['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
from itertools import repeat
import time
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm


def emos(params:list, y_true, member_values: pd.DataFrame, mu_vars: pd.DataFrame):
    a = params[0]
    b = np.array(params[1:-2])
    c = params[-2]
    d = params[-1]
    
    mu = a + mu_vars @ b
    spread2 = member_values.var(axis=1, ddof=1)
    sigma2 = c + d * spread2
    sigma2 = np.maximum(sigma2, 1e-6)
    loss = 0.5 * np.log(2 * np.pi * sigma2) + ((y_true - mu) ** 2) / (2 * sigma2)
    
    return np.mean(loss)


y_true = y_train.values
member_values = X_train[temps].values
mu_vars = X_train.drop(temps, axis=1).values

a = 0
b = np.array(list(repeat(0.1, mu_vars.shape[1])))
c = 1
d = 1

history = []
start_time = time.time()
def callback(xk):
    iteration = len(history) + 1
    current_loss = emos(xk, y_true, member_values, mu_vars)
    elapsed = time.time() - start_time

    history.append(current_loss)

    print(
        f"Iter {iteration:4d} | "
        f"Loss: {current_loss:.6f} | "
        f"Elapsed: {elapsed:.1f}s"
    )


pred_df = pd.DataFrame()

results = minimize(
    fun=emos,
    x0=[a, *b, c, d],
    args=(y_true, member_values, mu_vars),
    method="L-BFGS-B",
    callback=callback,
    #options = {'maxiter': 100}
)


y_true_test = y_test.values
member_values_test = X_test[temps].values
mu_vars_test = X_test.drop(temps, axis=1).values
    
best_params = results.x


a = best_params[0]
b = best_params[1:-2]
c = best_params[-2]
d = best_params[-1]

mu = a + mu_vars_test @ b
spread2 = member_values_test.var(axis=1, ddof=1)
sigma2 = c + d * spread2
sigma = np.sqrt(sigma2)
pred_df = pd.DataFrame(index = y_test.index)
pred_df["mu"] = mu
pred_df["sigma"] = sigma
pred_df["sigma2"] = sigma2
pred_df["obs"] = y_true_test

pred_df["pit"] = norm.cdf(pred_df.obs, loc=pred_df.mu, scale=pred_df.sigma)


In [ ]:
from scipy.stats import norm
best_params = results.x


a = best_params[0]
b = best_params[1:-2]
c = best_params[-2]
d = best_params[-1]

mu = a + mu_vars @ b
spread2 = member_values.var(axis=1, ddof=1)
sigma2 = c + d * spread2
sigma = np.sqrt(sigma2)
pred_df = pd.DataFrame(index = y_test.index)
pred_df["mu"] = mu
pred_df["sigma"] = sigma
pred_df["sigma2"] = sigma2
pred_df["obs"] = y_true

pred_df["pit"] = norm.cdf(pred_df.obs, loc=pred_df.mu, scale=pred_df.sigma)


pred_df.pit.plot(kind = 'hist', bins = 20, title = 'PIT Histogram')

In [ ]:
resid_std.plot(kind='hist', bins=50, title='Standardized Residuals')

In [ ]:
from itertools import repeat
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

X = bb[temps]
y = bb['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

def emos(params: list, y_true, member_values: pd.DataFrame, mu_vars: pd.DataFrame):
    a = params[0]
    b = np.array(params[1:-2])
    c = params[-2]
    d = params[-1]

    mu = a + mu_vars @ b
    spread2 = member_values.var(axis=1, ddof=1)
    sigma2 = c + d * spread2
    sigma2 = np.maximum(sigma2, 1e-6)
    loss = 0.5 * np.log(2 * np.pi * sigma2) + ((y_true - mu) ** 2) / (2 * sigma2)

    return np.mean(loss)


window_size = 40

a = 0
b = np.array(list(repeat(0.1, X_train.drop(temps, axis=1).shape[1])))
c = 1
d = 1

pred_df = pd.DataFrame(index=y_test.index)

# history available at each step
X_hist = X_train.copy()
y_hist = y_train.copy()

for i in range(len(X_test)):
    # rolling 40-row training window
    X_window = X_hist.iloc[-window_size:]
    y_window = y_hist.iloc[-window_size:]

    y_true = y_window.values
    member_values = X_window[temps].values
    mu_vars = X_window.drop(temps, axis=1).values

    history = []
    start_time = time.time()

    def callback(xk):
        iteration = len(history) + 1
        current_loss = emos(xk, y_true, member_values, mu_vars)
        elapsed = time.time() - start_time

        history.append(current_loss)

        print(
            f"Step {i+1:4d}/{len(X_test)} | "
            f"Iter {iteration:4d} | "
            f"Loss: {current_loss:.6f} | "
            f"Elapsed: {elapsed:.1f}s"
        )

    results = minimize(
        fun=emos,
        x0=[a, *b, c, d],
        args=(y_true, member_values, mu_vars),
        method="L-BFGS-B",
        callback=callback,
        # options={'maxiter': 100}
    )

    best_params = results.x

    a_step = best_params[0]
    b_step = best_params[1:-2]
    c_step = best_params[-2]
    d_step = best_params[-1]

    # predict 1 row ahead
    X_next = X_test.iloc[[i]]
    y_next = y_test.iloc[[i]]

    member_values_test = X_next[temps].values
    mu_vars_test = X_next.drop(temps, axis=1).values
    y_true_test = y_next.values

    mu = a_step + mu_vars_test @ b_step
    spread2 = member_values_test.var(axis=1, ddof=1)
    sigma2 = c_step + d_step * spread2
    sigma2 = np.maximum(sigma2, 1e-6)
    sigma = np.sqrt(sigma2)

    pred_df.loc[y_next.index, "mu"] = mu[0]
    pred_df.loc[y_next.index, "sigma"] = sigma[0]
    pred_df.loc[y_next.index, "sigma2"] = sigma2[0]
    pred_df.loc[y_next.index, "obs"] = y_true_test[0]

    # add newly observed row to history for next step
    X_hist = pd.concat([X_hist, X_next])
    y_hist = pd.concat([y_hist, y_next])

pred_df["pit"] = norm.cdf(pred_df.obs, loc=pred_df.mu, scale=pred_df.sigma)

In [ ]:
pred_df['cdf'] = norm.cdf(pred_df.obs, loc=pred_df.mu, scale=pred_df.sigma)
pred_df.dropna()


In [ ]:
pred_df.cdf.plot(kind='hist', bins=20, title='PIT Histogram')
